# 10 - PyTorch Baseline Training

**Pipeline:** Skin Cancer 3-Class Classification (NV / MEL / BCC)  
**Purpose:** Train two EfficientNetB0 baseline models: multiclass (NV/MEL/BCC) and binary (non_cancer vs cancer_risk=MEL+BCC).  

**Rules:**  
- Do not modify dataset files, manifests, splits, or images.  
- Do not evaluate on test set. Test evaluation is in File 11.  
- Model selection uses val_macro_f1 (multiclass) and val_pr_auc (binary).

---

## Section 0 - Imports and Environment

In [ ]:
import json
import os
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    precision_recall_fscore_support, recall_score,
    roc_auc_score, average_precision_score
)

CUDA_AVAILABLE = torch.cuda.is_available()
try:
    DEVICE         = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
    GPU_NAME       = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "N/A"
except AssertionError:
    DEVICE = torch.device("cpu")
    GPU_NAME = "N/A"
    CUDA_AVAILABLE = False

print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"CUDA        : {CUDA_AVAILABLE}")
print(f"GPU         : {GPU_NAME}")

torch       : 2.6.0+cu124
torchvision : 0.21.0+cu124
CUDA        : True
GPU         : NVIDIA GeForce RTX 4060 Ti
Device      : cuda


## Section 1 - Paths and Configuration

In [22]:
OUTPUT_ROOT        = Path(r"C:\SKIN CANCER v2\pipe output")
FINAL_DATASET_ROOT = Path(r"C:\SKIN CANCER v2\final DS")
PREPROC_DIR   = OUTPUT_ROOT / "preprocessing"
TRAINING_DIR  = OUTPUT_ROOT / "pytorch_training"

# Class mapping
CLASS_NAMES   = ["NV", "MEL", "BCC"]
CLASS_INDEX   = {"NV": 0, "MEL": 1, "BCC": 2}
BINARY_INDEX  = {"NV": 0, "MEL": 1, "BCC": 1}
BINARY_NAMES  = ["non_cancer", "cancer_risk"]

# Expected counts
EXPECTED_COUNTS = {
    "train": {"total": 14332, "NV": 8928, "MEL": 3144, "BCC": 2260},
    "val":   {"total":  3012, "NV": 1886, "MEL":  647, "BCC":  479},
    "test":  {"total":  3045, "NV": 1894, "MEL":  639, "BCC":  512},
}

# Training hyperparameters
IMG_SIZE     = 224
RESIZE_TO    = 256
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 5
NUM_WORKERS  = 0
PIN_MEMORY   = CUDA_AVAILABLE
RANDOM_SEED  = 42

# ImageNet normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Reproducibility
import random
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if CUDA_AVAILABLE:
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print("Config loaded.")
print(f"  EPOCHS={EPOCHS}  LR={LR}  BATCH_SIZE={BATCH_SIZE}  PATIENCE={EARLY_STOP_PATIENCE}")

Config loaded.
  EPOCHS=20  LR=0.0001  BATCH_SIZE=32  PATIENCE=5


## Section 2 - Create Output Folders

In [23]:
for d in [TRAINING_DIR,
          TRAINING_DIR / "multiclass",
          TRAINING_DIR / "binary"]:
    d.mkdir(parents=True, exist_ok=True)
print(f"Ready: {TRAINING_DIR}")

Ready: C:\SKIN CANCER v2\pipe output\pytorch_training


## Section 3 - Load Manifests

In [24]:
dfs = {}
for split in ["train", "val", "test"]:
    p = PREPROC_DIR / f"{split}_manifest_preprocessed.csv"
    if not p.exists():
        raise FileNotFoundError(f"Manifest not found: {p}\nRe-run File 08 first.")
    dfs[split] = pd.read_csv(p, low_memory=False)
    print(f"Loaded {split}: {len(dfs[split]):,} rows")

# Repair / add class_index
for split, df in dfs.items():
    if "class_index" not in df.columns or df["class_index"].isna().any():
        df["class_index"] = df["final_authoritative_label"].map(CLASS_INDEX)
    df["binary_label"] = df["final_authoritative_label"].map(BINARY_INDEX)

print("\n=== COUNT CHECK (informational) ===")
for split, df in dfs.items():
    ac = len(df); ec = EXPECTED_COUNTS[split]["total"]
    cc = df["final_authoritative_label"].value_counts()
    print(f"  {split}: {ac:,}  {'OK' if ac==ec else f'DIFF exp={ec:,}'}")
    for cls in CLASS_NAMES:
        a2 = int(cc.get(cls,0)); e2 = EXPECTED_COUNTS[split][cls]
        print(f"    {cls}: {a2:,}  {'OK' if a2==e2 else f'DIFF exp={e2:,}'}")

Loaded train: 14,332 rows
Loaded val: 3,012 rows
Loaded test: 3,045 rows

=== COUNT CHECK (informational) ===
  train: 14,332  OK
    NV: 8,928  OK
    MEL: 3,144  OK
    BCC: 2,260  OK
  val: 3,012  OK
    NV: 1,886  OK
    MEL: 647  OK
    BCC: 479  OK
  test: 3,045  OK
    NV: 1,894  OK
    MEL: 639  OK
    BCC: 512  OK


## Section 4 - Dataset Class and Transforms

In [25]:
# Transforms (identical to File 09)
train_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.RandomCrop(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(degrees=30),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize(RESIZE_TO),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class SkinLesionDataset(Dataset):
    MULTICLASS = {"NV": 0, "MEL": 1, "BCC": 2}
    BINARY     = {"NV": 0, "MEL": 1, "BCC": 1}

    def __init__(self, df, transform=None, label_mode="multiclass"):
        self.df         = df.reset_index(drop=True)
        self.transform  = transform
        self._lmap      = self.MULTICLASS if label_mode == "multiclass" else self.BINARY

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(str(row["preprocessed_full_path"])).convert("RGB")
        if self.transform: img = self.transform(img)
        lbl  = self._lmap.get(str(row["final_authoritative_label"]), -1)
        return img, torch.tensor(lbl, dtype=torch.long)


# Quick smoke test
_ds = SkinLesionDataset(dfs["train"].head(4), eval_transform, "multiclass")
_img, _lbl = _ds[0]
print(f"Dataset smoke test: img={_img.shape}  label={_lbl.item()}")
del _ds, _img, _lbl

Dataset smoke test: img=torch.Size([3, 224, 224])  label=0


## Section 5 - Build DataLoaders

In [26]:
def make_loader(df, transform, label_mode, shuffle):
    ds = SkinLesionDataset(df, transform, label_mode)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                      drop_last=False)

train_mc = make_loader(dfs["train"], train_transform, "multiclass", True)
val_mc   = make_loader(dfs["val"],   eval_transform,  "multiclass", False)
train_bi = make_loader(dfs["train"], train_transform, "binary",     True)
val_bi   = make_loader(dfs["val"],   eval_transform,  "binary",     False)

for name, ldr in [("train_mc",train_mc),("val_mc",val_mc),
                  ("train_bi",train_bi),("val_bi",val_bi)]:
    print(f"  {name:<12}: {len(ldr.dataset):>6,} imgs  {len(ldr):>4,} batches")

  train_mc    : 14,332 imgs   448 batches
  val_mc      :  3,012 imgs    95 batches
  train_bi    : 14,332 imgs   448 batches
  val_bi      :  3,012 imgs    95 batches


## Section 6 - Model Builder and Class Weights

In [27]:
def build_efficientnet_b0(n_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_feat, n_classes)
    )
    return model.to(DEVICE)


def compute_class_weights(label_series, n_classes):
    """Inverse-frequency weights normalised to n_classes."""
    counts = label_series.value_counts()
    n      = len(label_series)
    w = [n / (n_classes * counts.get(i, 1)) for i in range(n_classes)]
    return torch.FloatTensor(w).to(DEVICE)


# Multiclass weights
mc_weights = compute_class_weights(dfs["train"]["class_index"], 3)
print("Multiclass class weights:")
for cls, w in zip(CLASS_NAMES, mc_weights.cpu().tolist()):
    print(f"  {cls}: {w:.4f}")

# Binary weights
bi_weights = compute_class_weights(dfs["train"]["binary_label"], 2)
print("\nBinary class weights:")
for cls, w in zip(BINARY_NAMES, bi_weights.cpu().tolist()):
    print(f"  {cls}: {w:.4f}")

Multiclass class weights:
  NV: 0.5351
  MEL: 1.5195
  BCC: 2.1139

Binary class weights:
  non_cancer: 0.8026
  cancer_risk: 1.3261


## Section 7 - Training Infrastructure

In [28]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler is not None:
            with torch.amp.autocast("cuda", enabled=CUDA_AVAILABLE):
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(labels)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_multiclass(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out   = model(imgs)
        loss  = criterion(out, labels)
        total_loss += loss.item() * len(labels)
        preds = torch.argmax(out, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    y_true = np.array(all_labels); y_pred = np.array(all_preds)
    avg_loss = total_loss / len(loader.dataset)
    acc = float((y_pred == y_true).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    per_class = recall_score(y_true, y_pred, average=None,
                             labels=[0,1,2], zero_division=0)
    return {
        "val_loss":             round(avg_loss, 5),
        "val_accuracy":         round(acc, 5),
        "val_macro_precision":  round(float(prec), 5),
        "val_macro_recall":     round(float(rec),  5),
        "val_macro_f1":         round(float(f1),   5),
        "val_NV_recall":        round(float(per_class[0]), 5) if len(per_class)>0 else 0,
        "val_MEL_recall":       round(float(per_class[1]), 5) if len(per_class)>1 else 0,
        "val_BCC_recall":       round(float(per_class[2]), 5) if len(per_class)>2 else 0,
    }


@torch.no_grad()
def evaluate_binary(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out   = model(imgs)
        loss  = criterion(out, labels)
        total_loss += loss.item() * len(labels)
        probs = torch.softmax(out, dim=1)
        preds = torch.argmax(probs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    avg_loss = total_loss / len(loader.dataset)
    acc  = float((y_pred == y_true).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0,1], average=None, zero_division=0)
    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = 0.0
    try:
        pr_auc = average_precision_score(y_true, y_prob, pos_label=1)
    except Exception:
        pr_auc = 0.0
    return {
        "val_loss":             round(avg_loss, 5),
        "val_accuracy":         round(acc, 5),
        "val_precision_cancer": round(float(prec[1]), 5) if len(prec)>1 else 0,
        "val_recall_cancer":    round(float(rec[1]),  5) if len(rec)>1  else 0,
        "val_f1_cancer":        round(float(f1[1]),   5) if len(f1)>1   else 0,
        "val_roc_auc":          round(roc_auc, 5),
        "val_pr_auc":           round(pr_auc,  5),
    }


print("Training functions defined.")

Training functions defined.


In [29]:
def train_model(
    model_name, label_mode,
    train_loader, val_loader,
    class_weights, output_subdir,
    primary_metric_key
):
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}  ({label_mode})")
    print(f"  Primary metric: {primary_metric_key}")
    print(f"{'='*60}")

    n_classes = 3 if label_mode == "multiclass" else 2
    model     = build_efficientnet_b0(n_classes)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5,
        patience=2, min_lr=1e-7
    )
    scaler = torch.amp.GradScaler("cuda") if CUDA_AVAILABLE else None

    best_metric    = -1.0
    best_val_loss  = float("inf")
    best_epoch     = 0
    patience_count = 0
    history        = []
    t_start        = time.time()

    eval_fn = evaluate_multiclass if label_mode == "multiclass" else evaluate_binary

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler)
        val_metrics = eval_fn(model, val_loader, criterion)

        primary = val_metrics[primary_metric_key]
        cur_lr  = optimizer.param_groups[0]["lr"]
        scheduler.step(primary)

        is_best = primary > best_metric
        if is_best:
            best_metric   = primary
            best_val_loss = val_metrics["val_loss"]
            best_epoch    = epoch
            patience_count = 0
            torch.save({
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "optimizer":    optimizer.state_dict(),
                "best_metric":  best_metric,
                "val_loss":     best_val_loss,
                "label_mode":   label_mode,
            }, output_subdir / f"best_model_{model_name}.pt")
        else:
            patience_count += 1

        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "val_metrics": val_metrics,
            "label_mode":  label_mode,
        }, output_subdir / f"last_model_{model_name}.pt")

        row = {"epoch": epoch, "train_loss": round(train_loss,5),
               **val_metrics,
               "lr": cur_lr, "is_best": is_best,
               "epoch_time_s": round(time.time()-t0, 1)}
        history.append(row)

        tag = " << BEST" if is_best else ""
        print(f"  Ep {epoch:>2}/{EPOCHS}  "
              f"tr_loss={train_loss:.4f}  "
              f"val_loss={val_metrics['val_loss']:.4f}  "
              f"{primary_metric_key}={primary:.4f}  "
              f"lr={cur_lr:.2e}  "
              f"{int(time.time()-t0)}s{tag}")

        if patience_count >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch} "
                  f"(no improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

    elapsed = round(time.time() - t_start, 1)
    print(f"\nTraining complete in {elapsed}s")
    print(f"Best epoch: {best_epoch}  "
          f"best_{primary_metric_key}={best_metric:.5f}  "
          f"val_loss={best_val_loss:.5f}")

    return pd.DataFrame(history), best_epoch, best_metric, best_val_loss


print("train_model function defined.")

train_model function defined.


## Section 8 - Train Multiclass Model (NV / MEL / BCC)

In [30]:
mc_history, mc_best_epoch, mc_best_f1, mc_best_loss = train_model(
    model_name         = "multiclass",
    label_mode         = "multiclass",
    train_loader       = train_mc,
    val_loader         = val_mc,
    class_weights      = mc_weights,
    output_subdir      = TRAINING_DIR / "multiclass",
    primary_metric_key = "val_macro_f1",
)


  Training: multiclass  (multiclass)
  Primary metric: val_macro_f1
  Ep  1/20  tr_loss=0.5805  val_loss=0.5011  val_macro_f1=0.7483  lr=1.00e-04  146s << BEST
  Ep  2/20  tr_loss=0.4327  val_loss=0.4513  val_macro_f1=0.7734  lr=1.00e-04  151s << BEST
  Ep  3/20  tr_loss=0.3719  val_loss=0.4391  val_macro_f1=0.7859  lr=1.00e-04  148s << BEST
  Ep  4/20  tr_loss=0.3270  val_loss=0.4107  val_macro_f1=0.8003  lr=1.00e-04  149s << BEST
  Ep  5/20  tr_loss=0.2856  val_loss=0.4270  val_macro_f1=0.7944  lr=1.00e-04  151s
  Ep  6/20  tr_loss=0.2665  val_loss=0.4223  val_macro_f1=0.8028  lr=1.00e-04  152s << BEST
  Ep  7/20  tr_loss=0.2453  val_loss=0.4203  val_macro_f1=0.8095  lr=1.00e-04  149s << BEST
  Ep  8/20  tr_loss=0.2148  val_loss=0.4580  val_macro_f1=0.8015  lr=1.00e-04  146s
  Ep  9/20  tr_loss=0.2058  val_loss=0.4474  val_macro_f1=0.8169  lr=1.00e-04  151s << BEST
  Ep 10/20  tr_loss=0.1911  val_loss=0.4382  val_macro_f1=0.8136  lr=1.00e-04  146s
  Ep 11/20  tr_loss=0.1727  val_los

## Section 9 - Save Multiclass Artifacts

In [31]:
# Training history
mc_hist_path = TRAINING_DIR / "multiclass" / "training_history_multiclass.csv"
mc_history.to_csv(str(mc_hist_path), index=False)
print(f"Saved {mc_hist_path.name}  ({len(mc_history)} epochs)")

# Training config
mc_config = {
    "model": "efficientnet_b0", "label_mode": "multiclass",
    "n_classes": 3, "class_names": CLASS_NAMES,
    "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
    "epochs_run": len(mc_history), "epochs_max": EPOCHS,
    "lr": LR, "weight_decay": WEIGHT_DECAY,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "optimizer": "AdamW", "scheduler": "ReduceLROnPlateau",
    "class_weights_NV":  round(mc_weights[0].item(), 5),
    "class_weights_MEL": round(mc_weights[1].item(), 5),
    "class_weights_BCC": round(mc_weights[2].item(), 5),
    "best_epoch":     mc_best_epoch,
    "best_val_macro_f1": mc_best_f1,
    "best_val_loss":  mc_best_loss,
    "device": str(DEVICE), "gpu_name": GPU_NAME,
    "random_seed": RANDOM_SEED,
}
mc_cfg_path = TRAINING_DIR / "multiclass" / "training_config_multiclass.json"
with open(str(mc_cfg_path), "w") as f:
    json.dump(mc_config, f, indent=2)
print(f"Saved {mc_cfg_path.name}")

# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = mc_history["epoch"]
axes[0].plot(ep, mc_history["train_loss"], label="train", marker="o", ms=3)
axes[0].plot(ep, mc_history["val_loss"],   label="val",   marker="s", ms=3)
axes[0].axvline(mc_best_epoch, ls="--", color="gray", alpha=0.6, label=f"best ep {mc_best_epoch}")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(ep, mc_history["val_macro_f1"],  label="macro_f1",  marker="o", ms=3)
axes[1].plot(ep, mc_history["val_accuracy"],  label="accuracy",  marker="s", ms=3)
axes[1].axvline(mc_best_epoch, ls="--", color="gray", alpha=0.6)
axes[1].set_title("Val F1 / Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")

axes[2].plot(ep, mc_history["val_NV_recall"],  label="NV",  marker="o", ms=3)
axes[2].plot(ep, mc_history["val_MEL_recall"], label="MEL", marker="s", ms=3)
axes[2].plot(ep, mc_history["val_BCC_recall"], label="BCC", marker="^", ms=3)
axes[2].axvline(mc_best_epoch, ls="--", color="gray", alpha=0.6)
axes[2].set_title("Per-Class Recall"); axes[2].legend(); axes[2].set_xlabel("Epoch")

plt.suptitle("Multiclass Training — EfficientNetB0", fontsize=13)
plt.tight_layout()
mc_curve_path = TRAINING_DIR / "multiclass" / "training_curves_multiclass.png"
plt.savefig(str(mc_curve_path), dpi=100)
plt.close()
print(f"Saved {mc_curve_path.name}")

Saved training_history_multiclass.csv  (14 epochs)
Saved training_config_multiclass.json
Saved training_curves_multiclass.png


## Section 10 - Train Binary Model (non_cancer vs cancer_risk)

In [ ]:
bi_history, bi_best_epoch, bi_best_pr_auc, bi_best_loss = train_model(
    model_name         = "binary",
    label_mode         = "binary",
    train_loader       = train_bi,
    val_loader         = val_bi,
    class_weights      = bi_weights,
    output_subdir      = TRAINING_DIR / "binary",
    primary_metric_key = "val_pr_auc",
)


  Training: binary  (binary)
  Primary metric: val_pr_auc
  Ep  1/20  tr_loss=0.4033  val_loss=0.3565  val_pr_auc=0.8865  lr=1.00e-04  149s << BEST
  Ep  2/20  tr_loss=0.3288  val_loss=0.3194  val_pr_auc=0.9060  lr=1.00e-04  149s << BEST
  Ep  3/20  tr_loss=0.2880  val_loss=0.3533  val_pr_auc=0.8999  lr=1.00e-04  146s
  Ep  4/20  tr_loss=0.2600  val_loss=0.3451  val_pr_auc=0.9029  lr=1.00e-04  144s
  Ep  5/20  tr_loss=0.2330  val_loss=0.3412  val_pr_auc=0.9018  lr=1.00e-04  145s
  Ep  6/20  tr_loss=0.1958  val_loss=0.3580  val_pr_auc=0.9034  lr=5.00e-05  144s


## Section 11 - Save Binary Artifacts

In [ ]:
# Training history
bi_hist_path = TRAINING_DIR / "binary" / "training_history_binary.csv"
bi_history.to_csv(str(bi_hist_path), index=False)
print(f"Saved {bi_hist_path.name}  ({len(bi_history)} epochs)")

# Training config
bi_config = {
    "model": "efficientnet_b0", "label_mode": "binary",
    "n_classes": 2, "class_names": BINARY_NAMES,
    "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
    "epochs_run": len(bi_history), "epochs_max": EPOCHS,
    "lr": LR, "weight_decay": WEIGHT_DECAY,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "optimizer": "AdamW", "scheduler": "ReduceLROnPlateau",
    "class_weights_non_cancer":  round(bi_weights[0].item(), 5),
    "class_weights_cancer_risk": round(bi_weights[1].item(), 5),
    "best_epoch":     bi_best_epoch,
    "best_val_pr_auc":  bi_best_pr_auc,
    "best_val_loss":   bi_best_loss,
    "device": str(DEVICE), "gpu_name": GPU_NAME,
    "random_seed": RANDOM_SEED,
}
bi_cfg_path = TRAINING_DIR / "binary" / "training_config_binary.json"
with open(str(bi_cfg_path), "w") as f:
    json.dump(bi_config, f, indent=2)
print(f"Saved {bi_cfg_path.name}")

# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = bi_history["epoch"]
axes[0].plot(ep, bi_history["train_loss"], label="train", marker="o", ms=3)
axes[0].plot(ep, bi_history["val_loss"],   label="val",   marker="s", ms=3)
axes[0].axvline(bi_best_epoch, ls="--", color="gray", alpha=0.6, label=f"best ep {bi_best_epoch}")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(ep, bi_history["val_pr_auc"],  label="PR-AUC",  marker="o", ms=3)
axes[1].plot(ep, bi_history["val_roc_auc"], label="ROC-AUC", marker="s", ms=3)
axes[1].axvline(bi_best_epoch, ls="--", color="gray", alpha=0.6)
axes[1].set_title("Val AUC Metrics"); axes[1].legend(); axes[1].set_xlabel("Epoch")

axes[2].plot(ep, bi_history["val_recall_cancer"],    label="cancer recall",    marker="o", ms=3)
axes[2].plot(ep, bi_history["val_precision_cancer"], label="cancer precision", marker="s", ms=3)
axes[2].plot(ep, bi_history["val_f1_cancer"],        label="cancer F1",        marker="^", ms=3)
axes[2].axvline(bi_best_epoch, ls="--", color="gray", alpha=0.6)
axes[2].set_title("Cancer Class Metrics"); axes[2].legend(); axes[2].set_xlabel("Epoch")

plt.suptitle("Binary Training — EfficientNetB0", fontsize=13)
plt.tight_layout()
bi_curve_path = TRAINING_DIR / "binary" / "training_curves_binary.png"
plt.savefig(str(bi_curve_path), dpi=100)
plt.close()
print(f"Saved {bi_curve_path.name}")

## Section 12 - Save Combined Training Summary

In [ ]:
# Best multiclass epoch metrics
mc_best_row = mc_history[mc_history["epoch"] == mc_best_epoch].iloc[0]
bi_best_row = bi_history[bi_history["epoch"] == bi_best_epoch].iloc[0]

summary_df = pd.DataFrame([
    {"model": "multiclass",
     "best_epoch":          mc_best_epoch,
     "best_primary_metric": mc_best_f1,
     "primary_metric_name": "val_macro_f1",
     "best_val_loss":       mc_best_loss,
     "best_val_accuracy":   float(mc_best_row["val_accuracy"]),
     "best_val_MEL_recall": float(mc_best_row["val_MEL_recall"]),
     "best_val_BCC_recall": float(mc_best_row["val_BCC_recall"]),
     "best_val_NV_recall":  float(mc_best_row["val_NV_recall"]),
     "epochs_run": len(mc_history), "device": str(DEVICE)},
    {"model": "binary",
     "best_epoch":           bi_best_epoch,
     "best_primary_metric":  bi_best_pr_auc,
     "primary_metric_name":  "val_pr_auc",
     "best_val_loss":        bi_best_loss,
     "best_val_accuracy":    float(bi_best_row["val_accuracy"]),
     "best_val_cancer_recall": float(bi_best_row["val_recall_cancer"]),
     "best_val_roc_auc":     float(bi_best_row["val_roc_auc"]),
     "best_val_pr_auc":      float(bi_best_row["val_pr_auc"]),
     "epochs_run": len(bi_history), "device": str(DEVICE)},
])
summary_path = TRAINING_DIR / "pytorch_training_summary.csv"
summary_df.to_csv(str(summary_path), index=False)
print(f"Saved {summary_path.name}")
display(summary_df)

## Section 13 - Output File Verification

In [ ]:
required_files = [
    TRAINING_DIR / "multiclass" / "best_model_multiclass.pt",
    TRAINING_DIR / "multiclass" / "last_model_multiclass.pt",
    TRAINING_DIR / "multiclass" / "training_history_multiclass.csv",
    TRAINING_DIR / "multiclass" / "training_curves_multiclass.png",
    TRAINING_DIR / "multiclass" / "training_config_multiclass.json",
    TRAINING_DIR / "binary"     / "best_model_binary.pt",
    TRAINING_DIR / "binary"     / "last_model_binary.pt",
    TRAINING_DIR / "binary"     / "training_history_binary.csv",
    TRAINING_DIR / "binary"     / "training_curves_binary.png",
    TRAINING_DIR / "binary"     / "training_config_binary.json",
    TRAINING_DIR / "pytorch_training_summary.csv",
]
print("Output file verification:")
all_ok = True
for p in required_files:
    exists = p.exists(); size = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {p.name:<45} {size:>12,} bytes")
    if not exists: all_ok = False
print("\nAll required files present." if all_ok else "\nWARNING: missing files.")

## Section 14 - Final Summary (Copy-Paste Ready)

In [ ]:
from IPython.display import display

print("=" * 70)
print("  10_pytorch_training -- FINAL SUMMARY")
print("=" * 70)

print(f"\n 1. torch version         : {torch.__version__}")
print(f" 2. CUDA available        : {CUDA_AVAILABLE}")
print(f" 3. GPU name              : {GPU_NAME}")
print(f" 4. Device used           : {DEVICE}")

print(f"\n 5. Train / val row counts:")
for split in ["train", "val"]:
    print(f"    {split}: {len(dfs[split]):,}")

print(f"\n 6. Train / val class counts:")
for split in ["train", "val"]:
    cc = dfs[split]["final_authoritative_label"].value_counts()
    print(f"    {split}: NV={int(cc.get('NV',0)):,}  "
          f"MEL={int(cc.get('MEL',0)):,}  BCC={int(cc.get('BCC',0)):,}")

print(f"\n 7. Multiclass class weights:")
for cls, w in zip(CLASS_NAMES, mc_weights.cpu().tolist()):
    print(f"    {cls}: {w:.5f}")

print(f"\n 8. Binary class weights:")
for cls, w in zip(BINARY_NAMES, bi_weights.cpu().tolist()):
    print(f"    {cls}: {w:.5f}")

print(f"\n 9. Multiclass best epoch  : {mc_best_epoch}")
print(f"10. Multiclass best val_macro_f1 : {mc_best_f1:.5f}")
print(f"11. Multiclass best val_loss     : {mc_best_loss:.5f}")
mc_best_mel = float(mc_history.loc[mc_history["epoch"]==mc_best_epoch, "val_MEL_recall"].values[0])
print(f"12. Multiclass best MEL recall   : {mc_best_mel:.5f}")

print(f"\n13. Binary best epoch      : {bi_best_epoch}")
print(f"14. Binary best val_pr_auc  : {bi_best_pr_auc:.5f}")
bi_best_rec = float(bi_history.loc[bi_history["epoch"]==bi_best_epoch, "val_recall_cancer"].values[0])
print(f"15. Binary best cancer recall : {bi_best_rec:.5f}")
print(f"16. Binary best val_loss      : {bi_best_loss:.5f}")

print(f"\n17. Output file verification:")
for p in required_files:
    exists = p.exists(); size = p.stat().st_size if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"    [{status}] {p.name:<45} {size:>10,} bytes")

print(f"\n18. Test set used for training    : False")
print(f"19. Test set used for model selection : False")
print(f"20. Images or manifests modified  : False")

print("\nMulticlass epoch history (last 5 epochs):")
mc_cols = ["epoch","train_loss","val_loss","val_macro_f1",
           "val_MEL_recall","val_BCC_recall","is_best"]
display(mc_history[[c for c in mc_cols if c in mc_history.columns]].tail(5))

print("\nBinary epoch history (last 5 epochs):")
bi_cols = ["epoch","train_loss","val_loss","val_pr_auc",
           "val_recall_cancer","val_roc_auc","is_best"]
display(bi_history[[c for c in bi_cols if c in bi_history.columns]].tail(5))

print("=" * 70)

## Section 15 - Completion Summary

**Section 10 - PyTorch Baseline Training is complete.**

What was accomplished:
- Two EfficientNetB0 models trained from ImageNet pretrained weights.
- Class-weighted CrossEntropyLoss to handle class imbalance.
- ReduceLROnPlateau scheduler with patience=2 on primary validation metric.
- Early stopping with patience=5.
- Best model checkpointed on `val_macro_f1` (multiclass) and `val_pr_auc` (binary).
- Training history and curves saved for both models.
- Test set was never accessed.

**What was deliberately deferred:**
- Test set evaluation
- Decision threshold tuning
- Confusion matrices on test set
- Calibration
- Grad-CAM / explainability

**Next notebook:** `11_evaluation.ipynb`  
Load `best_model_multiclass.pt` and `best_model_binary.pt`, evaluate on the test set, compute confusion matrices, tune thresholds, and report final performance metrics.